# ARC26 dual puzzle-LoRA 48x2 / q9 16x2 submission

Controlled self-ensemble derived from the 32.08 Vanilla V2 Version 4 launcher. Two independently seeded rank-256 per-puzzle LoRAs run concurrently: branch A on GPUs 0-1 and branch B on GPUs 2-3. Each LoRA trains on 8 geometries x 6 colour/order augmentations (48 optimizer steps) and decodes 8 geometries x 2 colour/order views (16 prompts). Candidate grids from both branches are pooled and ranked once with the unchanged score_kgmon selector. There is no LoRA-weight averaging, token-logit averaging, global LoRA, adaptive search, or silent one-branch fallback.


In [ ]:
import os

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["OMP_NUM_THREADS"] = "12"


In [ ]:
MODE = "submit_competition"  # validation | submit_competition

CODE_DATASET_ROOT = "/kaggle/input/datasets/yuvraj/arc2026"
MODEL_PATH = "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
COMP_ROOT = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"

VALIDATION_KEYS = None
BRANCH_NPROCS = 2
DFS_PROB_THRESHOLD = 0.2
UNSLOTH_MULTITOKEN_REPEAT_LEN = 9
TRAIN_COLOR_PERMUTATIONS = 6
EVAL_COLOR_PERMUTATIONS = 2
BRANCHES = [
    dict(name="a", cuda_offset=0, train_seed=1, eval_seed=2, trainer_seed=42),
    dict(name="b", cuda_offset=2, train_seed=1001, eval_seed=1002, trainer_seed=1042),
]
SELECTION_ALGORITHM = "score_kgmon"
PROFILE_TIMINGS = True

VALIDATION_END_TIME_HOURS = 2.5
SUBMIT_COMPETITION_END_TIME_HOURS = 11 + 50 / 60
RESET_RUN_ARTIFACTS = True

WORK_NOTEBOOK_ROOT = "/kaggle/working/arc2026_run_dual_lora_48x2_q9_16x2"
WORK_CODE_DIR = "/kaggle/working/arc2026_run_dual_lora_48x2_q9_16x2/ARC-AGI1/qwen_baseline"
WRITABLE_UNSLOTH_PARENT = "/kaggle/working/dual_lora_48x2_q9_16x2_stack"
EXPECTED_CODE_HASHES = {'starter.py': '6b884267cf2859e47a050e3753cd74b9cc5eddff013ff8aacaae25d81463d267', 'arc_solver.py': '15b689fdc1496ef47428645f92bcbb70fc2dae724d16a3bb21e87927df956823'}
MANIFEST_PATH = "/kaggle/working/dual_lora_48x2_manifest.json"


In [ ]:
import os
from pathlib import Path


def _truthy_env(name: str) -> bool:
    value = os.getenv(name)
    if value is None:
        return False
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


IS_KAGGLE_RERUN = _truthy_env("KAGGLE_IS_COMPETITION_RERUN")
assert MODE in {"validation", "submit_competition"}
EFFECTIVE_MODE = "submit_competition" if IS_KAGGLE_RERUN else MODE

EVAL_CHALLENGES = f"{COMP_ROOT}/arc-agi_evaluation_challenges.json"
EVAL_SOLUTIONS = f"{COMP_ROOT}/arc-agi_evaluation_solutions.json"
TEST_CHALLENGES = f"{COMP_ROOT}/arc-agi_test_challenges.json"

if IS_KAGGLE_RERUN:
    TEST_PATH = TEST_CHALLENGES
    SOLUTION_PATH = None
    OUTPUT_DIR_A = "/kaggle/working/inference_outputs_dual_lora_branch_a"
    OUTPUT_DIR_B = "/kaggle/working/inference_outputs_dual_lora_branch_b"
    SUBMISSION_PATH = "/kaggle/working/submission.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = None
    END_TIME_HOURS = SUBMIT_COMPETITION_END_TIME_HOURS
    RUN_INFERENCE = True
elif MODE == "validation":
    TEST_PATH = EVAL_CHALLENGES
    SOLUTION_PATH = EVAL_SOLUTIONS
    OUTPUT_DIR_A = "/kaggle/working/inference_outputs_dual_lora_branch_a_validation"
    OUTPUT_DIR_B = "/kaggle/working/inference_outputs_dual_lora_branch_b_validation"
    SUBMISSION_PATH = "/kaggle/working/validation_submission_unsloth_q9.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = VALIDATION_KEYS
    END_TIME_HOURS = VALIDATION_END_TIME_HOURS
    RUN_INFERENCE = True
else:
    TEST_PATH = TEST_CHALLENGES
    SOLUTION_PATH = None
    OUTPUT_DIR_A = "/kaggle/working/inference_outputs_dual_lora_branch_a_shortcut"
    OUTPUT_DIR_B = "/kaggle/working/inference_outputs_dual_lora_branch_b_shortcut"
    SUBMISSION_PATH = "/kaggle/working/submission.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = None
    END_TIME_HOURS = 0.0
    RUN_INFERENCE = False

print("mode_requested =", MODE)
print("is_kaggle_rerun =", IS_KAGGLE_RERUN)
print("effective_mode =", EFFECTIVE_MODE)
print("test_path =", TEST_PATH)
print("output_dirs =", OUTPUT_DIR_A, OUTPUT_DIR_B)
print("submission_path =", SUBMISSION_PATH)
print("selected_keys =", SELECTED_KEYS)
print("end_time_hours =", END_TIME_HOURS)
print("run_inference =", RUN_INFERENCE)


In [ ]:
import importlib.util
import os
import shutil
import sys
from pathlib import Path

assert Path(CODE_DATASET_ROOT).exists(), f"Missing code dataset root: {CODE_DATASET_ROOT}"
assert Path(MODEL_PATH).exists(), f"Missing model path: {MODEL_PATH}"
assert Path(TEST_PATH).exists(), f"Missing challenge path: {TEST_PATH}"
assert Path(os.environ["TRITON_PTXAS_PATH"]).exists(), os.environ["TRITON_PTXAS_PATH"]
if SOLUTION_PATH is not None:
    assert Path(SOLUTION_PATH).exists(), f"Missing solution path: {SOLUTION_PATH}"

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"

os.chdir("/kaggle/working")
print("setup cwd =", os.getcwd())

if RESET_RUN_ARTIFACTS:
    for path in [WORK_NOTEBOOK_ROOT, OUTPUT_DIR_A, OUTPUT_DIR_B, WRITABLE_UNSLOTH_PARENT]:
        shutil.rmtree(path, ignore_errors=True)
    try:
        Path(SUBMISSION_PATH).unlink()
    except FileNotFoundError:
        pass
    for path in Path("/kaggle/working").glob("worker_train_dual_*"): 
        if path.is_file():
            path.unlink()

for module_name in ["unsloth", "transformers", "torch"]:
    spec = importlib.util.find_spec(module_name)
    print(module_name, spec.origin if spec else "MISSING")


In [ ]:
import os
import shutil
from pathlib import Path

src = Path(CODE_DATASET_ROOT)
dst = Path(WORK_NOTEBOOK_ROOT)
shutil.copytree(src, dst)

observed_code_hashes = {
    name: __import__('hashlib').sha256(Path(WORK_CODE_DIR, name).read_bytes()).hexdigest()
    for name in EXPECTED_CODE_HASHES
}
if observed_code_hashes != EXPECTED_CODE_HASHES:
    raise RuntimeError({"expected_code_hashes": EXPECTED_CODE_HASHES, "observed": observed_code_hashes})

required_files = [
    "starter.py",
    "arc_solver.py",
    "arc_search_multitoken.py",
    "patch_unsloth_qwen3_multitoken.py",
]
for name in required_files:
    assert Path(WORK_CODE_DIR, name).is_file(), (
        f"arc2026 is stale: missing {name}"
    )

starter_source = Path(WORK_CODE_DIR, "starter.py").read_text()
solver_source = Path(WORK_CODE_DIR, "arc_solver.py").read_text()
assert "--use-unsloth-multitoken-dfs" in starter_source
assert "--eval-color-permutations" in starter_source
assert "--train-color-permutations" in starter_source
assert "--train-augmentation-seed" in starter_source
assert "--eval-augmentation-seed" in starter_source
assert "--trainer-seed" in starter_source
assert "--sentinel-tag" in starter_source
assert "UNSLOTH_COMPILE_LOCATION" in starter_source, "arc2026 starter.py lacks the worker import-race guard"
assert '"embed_tokens"' in solver_source and '"lm_head"' in solver_source
assert "inference_turbo_dfs_multitoken" in solver_source
print("arc2026 dual-LoRA production preflight passed")
print("code_hashes =", observed_code_hashes)
print("work_code_dir =", WORK_CODE_DIR)


In [ ]:
spec = importlib.util.find_spec("unsloth")
assert spec is not None and spec.submodule_search_locations
mounted_unsloth = Path(next(iter(spec.submodule_search_locations)))
qwen_source = (mounted_unsloth / "models" / "qwen3.py").read_text()
assert "A = flash_attn_func(Qnn, Knn, Vnn)" in qwen_source

writable_parent = Path(WRITABLE_UNSLOTH_PARENT)
writable_unsloth = writable_parent / "unsloth"
shutil.copytree(mounted_unsloth, writable_unsloth)

sys.path.insert(0, WORK_CODE_DIR)
from patch_unsloth_qwen3_multitoken import PATCH_MARKER, patch_unsloth

changed = patch_unsloth(writable_unsloth)
assert PATCH_MARKER in (writable_unsloth / "models" / "qwen3.py").read_text()
print("writable_unsloth =", writable_unsloth)
print("patched =", [str(path) for path in changed])

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = str(writable_parent) + os.pathsep + RUN_ENV.get("PYTHONPATH", "")


In [ ]:
import json
import subprocess
import sys
import time


def branch_command(branch, output_dir, absolute_end_time):
    command = [
        sys.executable,
        "starter.py",
        "--test-path", TEST_PATH,
        "--model-path", MODEL_PATH,
        "--output-dir", output_dir,
        "--nprocs", str(BRANCH_NPROCS),
        "--cuda-device-offset", str(branch["cuda_offset"]),
        "--use-unsloth-multitoken-dfs",
        "--unsloth-multitoken-repeat-len", str(UNSLOTH_MULTITOKEN_REPEAT_LEN),
        "--dfs-prob-threshold", str(DFS_PROB_THRESHOLD),
        "--train-color-permutations", str(TRAIN_COLOR_PERMUTATIONS),
        "--eval-color-permutations", str(EVAL_COLOR_PERMUTATIONS),
        "--train-augmentation-seed", str(branch["train_seed"]),
        "--eval-augmentation-seed", str(branch["eval_seed"]),
        "--trainer-seed", str(branch["trainer_seed"]),
        "--sentinel-tag", f"dual_{branch['name']}",
        "--end-time", str(absolute_end_time),
    ]
    if PROFILE_TIMINGS:
        command.append("--profile-timings")
    if SELECTED_KEYS is not None:
        command.extend(["--keys-json", json.dumps(SELECTED_KEYS)])
    return command


if RUN_INFERENCE:
    absolute_end_time = time.time() + END_TIME_HOURS * 3600
    dual_started_at = time.perf_counter()
    branch_specs = [
        (BRANCHES[0], OUTPUT_DIR_A),
        (BRANCHES[1], OUTPUT_DIR_B),
    ]
    processes = []
    for branch, output_dir in branch_specs:
        command = branch_command(branch, output_dir, absolute_end_time)
        print(f"starting branch {branch['name']}:", " ".join(command), flush=True)
        processes.append((branch["name"], subprocess.Popen(command, cwd=WORK_CODE_DIR, env=RUN_ENV)))

    pending = dict(processes)
    while pending:
        for name, process in list(pending.items()):
            return_code = process.poll()
            if return_code is None:
                continue
            del pending[name]
            print(f"branch {name} exit_code={return_code}", flush=True)
            if return_code != 0:
                for other in pending.values():
                    other.terminate()
                for other in pending.values():
                    other.wait()
                raise subprocess.CalledProcessError(return_code, f"dual-LoRA branch {name}")
        if pending:
            time.sleep(5)
    dual_wall_seconds = time.perf_counter() - dual_started_at
    print("dual_branch_wall_seconds =", dual_wall_seconds, flush=True)
else:
    dual_wall_seconds = 0.0
    print("save-version shortcut: full inference runs only during the competition rerun")


In [ ]:
import json
import sys
from pathlib import Path

if WORK_CODE_DIR not in sys.path:
    sys.path.insert(0, WORK_CODE_DIR)

from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, score_kgmon


data = ArcDataset.from_file(TEST_PATH)
if EFFECTIVE_MODE == "validation" and SOLUTION_PATH is not None:
    data = data.load_replies(SOLUTION_PATH)
split_data = data.split_multi_replies()
expected_outputs = set(split_data.keys)

if not RUN_INFERENCE:
    submission = data.get_submission()
    Path(SUBMISSION_PATH).write_text(json.dumps(submission))
    print("save-version shortcut submission only; competition rerun performs dual-LoRA inference")
    print("submission_path =", SUBMISSION_PATH)
else:
    branch_decoders = {}
    branch_missing = {}
    for branch_name, output_dir in (("a", OUTPUT_DIR_A), ("b", OUTPUT_DIR_B)):
        decoder = ArcDecoder(split_data, n_guesses=2)
        path = Path(output_dir)
        if path.exists() and any(path.iterdir()):
            decoder.load_decoded_results(output_dir, run_name=f".branch_{branch_name}")
        branch_decoders[branch_name] = decoder
        branch_missing[branch_name] = sorted(expected_outputs - set(decoder.decoded_results))

    manifest = {
        "recipe": "dual_lora_48x2_q9_16x2",
        "expected_outputs": len(expected_outputs),
        "branch_output_counts": {
            name: len(decoder.decoded_results)
            for name, decoder in branch_decoders.items()
        },
        "branch_missing": branch_missing,
        "train_steps_per_branch": 8 * TRAIN_COLOR_PERMUTATIONS,
        "inference_views_per_branch": 8 * EVAL_COLOR_PERMUTATIONS,
        "dual_branch_wall_seconds": dual_wall_seconds,
        "branches": BRANCHES,
    }
    Path(MANIFEST_PATH).write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
    print(json.dumps(manifest, indent=2, sort_keys=True))

    if any(branch_missing.values()):
        raise RuntimeError(
            "Dual-LoRA run incomplete; refusing one-branch or placeholder submission. "
            f"See {MANIFEST_PATH}"
        )

    combined = ArcDecoder(split_data, n_guesses=2)
    combined.load_decoded_results(OUTPUT_DIR_A, run_name=".branch_a")
    combined.load_decoded_results(OUTPUT_DIR_B, run_name=".branch_b")
    selected = combined.run_selection_algo(score_kgmon)
    submission = data.get_submission(selected)
    Path(SUBMISSION_PATH).write_text(json.dumps(submission))

    print("combined_output_keys =", len(combined.decoded_results))
    print("submission_path =", SUBMISSION_PATH)
    print("submission_tasks =", len(submission))

    if EFFECTIVE_MODE == "validation" and SOLUTION_PATH is not None:
        for branch_name, decoder in branch_decoders.items():
            print(f"branch_{branch_name}_validation")
            decoder.benchmark_selection_algos()
        combined.benchmark_selection_algos()
        print("validation_score =", data.validate_submission(submission))
